In [1]:
# --- Core Python ---
import os
import sys
from pathlib import Path
import h5py
import re

# Ensure notebook-local helper modules are importable
_NB_DIR = Path.cwd()
print(f"currently in {_NB_DIR}")
if (_NB_DIR / "resp_helper_functions.py").exists():
    print("helper functions exists in working directory, adding to sys.path")
    sys.path.insert(0, str(_NB_DIR))
else:
    # Fallback when notebook is launched from workspace root
    _NB_DIR = Path("notebooks/thomas_notebooks/resp_classification").resolve()
    if (_NB_DIR / "resp_helper_functions.py").exists():
        sys.path.insert(0, str(_NB_DIR))
    print("Helper functions didn't exist in working directory, resolving and adding to sys.path")

# --- Numerical & data analysis ---
import numpy as np
import pandas as pd

# --- Plotting ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Signal processing ---
from scipy.signal import butter, filtfilt, resample_poly, find_peaks

# --- Statistics ---
from scipy.stats import wilcoxon

# --- Machine learning & metrics ---
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

# --- Specialized neurophysiology tools ---
import neurokit2 as nk

# Shared signal + breathmetrics utilities only (rank pipeline lives in this notebook)
from resp_helper_functions import (
    fit_bm_session,
    sample_random_nonoverlapping_windows,
    extract_features_from_windows,
)

from math import gcd


# --- Pandas display settings ---
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 1000)


currently in c:\Users\thoma\Code\ResearchCode\respiratory_pilot
Helper functions didn't exist in working directory, resolving and adding to sys.path
Libraries loaded successfully


In [2]:
cagemate_rank_h5s = r"C:\Users\thoma\UF Dropbox\Thomas Heeps\Padilla-Coreano Lab\2025\BLA_ChR_resp_pilot1\BLA_resp_ChR_cm_hc\h5_outputs"

In [3]:
_h5_dir = Path(cagemate_rank_h5s)
rank_h5_files = sorted(p.name for p in _h5_dir.iterdir() if p.is_file())
print(f"{len(rank_h5_files)} files in {cagemate_rank_h5s}:\n")
for name in rank_h5_files:
    print(name)

25 files in C:\Users\thoma\UF Dropbox\Thomas Heeps\Padilla-Coreano Lab\2025\BLA_ChR_resp_pilot1\BLA_resp_ChR_cm_hc\h5_outputs:

1_1_d_cm_1_2_i_20260106_134350.h5
1_1_i_cm_1_3_s_20260107_163611.h5
1_2_d_cm_1_3_s_20260108_114416.h5
1_3_s_1_2_d_20260107_165340.h5
1_3_s_cm_1_1_i_20260106_122823.h5
2_1_d_cm_2.3_i_20260106_105109.h5
2_1_d_cm_2_2_s_20260108_112426.h5
2_2_s_cm_2_1_d_20260108_120453.h5
2_2_s_cm_2_1_d_20260108_120453.rec.h5
2_2_s_cm_2_3_i_20260106_132549.h5
3_1_i_cm_3_2_d_20260106_115157.h5
3_1_i_cm_3_3_s_20260107_123230.h5
3_2_d_cm_3_3_s_20260107_140924.h5
3_3_s_cm_3_1_i_20260106_124546.h5
3_3_s_cm_3_2_d_20260107_171330.h5
4_1_s_cm_4_2_d_20260107_160439.h5
4_2_d_cm_4_1_s_20260107_150836.h5
4_3_d_cm_4_1_s_20260107_113120.h5
5_1_i_cm_5_2_d_20260106_120946.h5
5_1_i_cm_5_3_s_20260107_132309.h5
6_2_d_cm_6_3_s_20260107_154939.h5
7_1_d_cm_7_2_s_20260106_111330.h5
7_2_s_cm_7_1_d_20260106_130431.h5
8_1_s_cm_8_2_d_20260107_152812.h5
8_2_s_cm_8_1_d_20260106_113241.h5


In [4]:
rename_map = {
    "1_1_d_cm_1_2_i_20260106_134350.h5": "1_1_2_i_cm_1_2_d_20260106_134350_SUB.h5",
    "1_1_i_cm_1_3_s_20260107_163611.h5": "1_1_2_i_cm_1_3_s_20260107_163611_DOM.h5",
    "1_2_d_cm_1_3_s_20260108_114416.h5": "1_2_2_d_cm_1_3_s_20260108_114416_DOM.h5",
    "1_3_s_cm_1_1_i_20260106_122823.h5": "1_3_2_s_cm_1_1_i_20260106_122823_DOM.h5",
    "1_3_s_1_2_d_20260107_165340.h5": "1_3_2_s_cm_1_2_d_20260107_165340_SUB.h5",
    "2_1_d_cm_2.3_i_20260106_105109.h5": "2_1_2_d_cm_2.3_i_20260106_105109_SUB.h5",
    "2_1_d_cm_2_2_s_20260108_112426.h5": "2_1_2_d_cm_2_2_s_20260108_112426_DOM.h5",
    "2_2_s_cm_2_1_d_20260108_120453.h5": "2_2_2_s_cm_2_1_d_20260108_120453_SUB.h5",
    "2_2_s_cm_2_3_i_20260106_132549.h5": "2_2_2_s_cm_2_3_i_20260106_132549_DOM.h5",
    "3_1_i_cm_3_2_d_20260106_115157.h5": "3_1_2_i_cm_3_2_d_20260106_115157_SUB.h5",
    "3_1_i_cm_3_3_s_20260107_123230.h5": "3_1_2_i_cm_3_3_s_20260107_123230_DOM.h5",
    "3_2_d_cm_3_3_s_20260107_140924.h5": "3_2_2_d_cm_3_3_s_20260107_140924_DOM.h5",
    "3_3_s_cm_3_1_i_20260106_124546.h5": "3_3_2_s_cm_3_1_i_20260106_124546_DOM.h5",
    "3_3_s_cm_3_2_d_20260107_171330.h5": "3_3_2_s_cm_3_2_d_20260107_171330_SUB.h5",
    "4_1_s_cm_4_2_d_20260107_160439.h5": "4_1_2_s_cm_4_2_d_20260107_160439_SUB.h5",
    "4_2_d_cm_4_1_s_20260107_150836.h5": "4_2_2_d_cm_4_1_s_20260107_150836_DOM.h5",
    "4_3_d_cm_4_1_s_20260107_113120.h5": "4_3_2_d_cm_4_1_s_20260107_113120_DOM.h5",
    "5_1_i_cm_5_2_d_20260106_120946.h5": "5_1_2_i_cm_5_2_d_20260106_120946_SUB.h5",
    "5_1_i_cm_5_3_s_20260107_132309.h5": "5_1_2_i_cm_5_3_s_20260107_132309_DOM.h5",
    "6_2_d_cm_6_3_s_20260107_154939.h5": "6_2_2_d_cm_6_3_s_20260107_154939_DOM.h5",
    "7_1_d_cm_7_2_s_20260106_111330.h5": "7_1_2_d_cm_7_2_s_20260106_111330_DOM.h5",
    "7_2_s_cm_7_1_d_20260106_130431.h5": "7_2_2_s_cm_7_1_d_20260106_130431_SUB.h5",
    "8_1_s_cm_8_2_d_20260107_152812.h5": "8_1_2_s_cm_8_2_d_20260107_152812_SUB.h5",
    "8_2_s_cm_8_1_d_20260106_113241.h5": "8_2_2_s_cm_8_1_d_20260106_113241_SUB.h5",
}

In [5]:
import h5py

h5_path = r"C:\Users\thoma\UF Dropbox\Thomas Heeps\Padilla-Coreano Lab\2025\BLA_ChR_resp_pilot1\BLA_resp_ChR_cm_hc\h5_outputs\1_1_d_cm_1_2_i_20260106_134350.h5"

def print_h5_structure(name, obj):
    print(name)

with h5py.File(h5_path, "r") as f:
    print("=== TOP LEVEL KEYS ===")
    for key in f.keys():
        print(key)

    print("\n=== FULL STRUCTURE ===")
    f.visititems(print_h5_structure)

=== TOP LEVEL KEYS ===
analog
dio
resp_clean
time

=== FULL STRUCTURE ===
analog
analog/1_1_d_cm_1_2_i_20260106_134350.analog_ECU_Ain1
analog/1_1_d_cm_1_2_i_20260106_134350.timestamps
dio
resp_clean
resp_clean/signal
resp_clean/time
time
time/1_1_d_cm_1_2_i_20260106_134350.timestamps


In [6]:
import h5py
import numpy as np
from scipy.signal import resample_poly
from math import gcd  # <-- this was missing


def load_clean_resp_signal_cagemate_rank(h5_path, target_rate=400):
    """
    Load cleaned respiration signal from the cagemate-rank H5 files.

    Expected H5 structure:
        resp_clean/
            signal
            time
    """
    try:
        with h5py.File(h5_path, "r") as f:
            if "resp_clean" not in f:
                raise KeyError("Missing group 'resp_clean'")
            if "signal" not in f["resp_clean"]:
                raise KeyError("Missing dataset 'resp_clean/signal'")
            if "time" not in f["resp_clean"]:
                raise KeyError("Missing dataset 'resp_clean/time'")

            raw_signal = np.asarray(f["resp_clean"]["signal"][:]).squeeze()
            raw_time = np.asarray(f["resp_clean"]["time"][:]).squeeze()

        if raw_signal.ndim != 1:
            raise ValueError(f"resp_clean/signal is not 1D. shape={raw_signal.shape}")
        if raw_time.ndim != 1:
            raise ValueError(f"resp_clean/time is not 1D. shape={raw_time.shape}")
        if len(raw_signal) == 0 or len(raw_time) == 0:
            raise ValueError("Signal or time array is empty")

        # Trim to shortest if slight mismatch
        if len(raw_signal) != len(raw_time):
            min_len = min(len(raw_signal), len(raw_time))
            print(
                f"Warning: signal/time length mismatch in {h5_path}. "
                f"Trimming from signal={len(raw_signal)}, time={len(raw_time)} to {min_len}."
            )
            raw_signal = raw_signal[:min_len]
            raw_time = raw_time[:min_len]

        # Estimate original sampling rate from time vector
        dt = np.diff(raw_time)
        dt = dt[np.isfinite(dt)]

        if len(dt) == 0:
            raise ValueError("Could not estimate sampling rate from time vector")

        median_dt = np.median(dt)
        if median_dt <= 0:
            raise ValueError(f"Non-positive median dt detected: {median_dt}")

        original_fs = 1.0 / median_dt

        # Resample if needed
        if target_rate is not None and not np.isclose(original_fs, target_rate, rtol=1e-3):
            target_rate_int = int(round(target_rate))
            original_fs_int = int(round(original_fs))

            common_div = gcd(target_rate_int, original_fs_int)
            up = target_rate_int // common_div
            down = original_fs_int // common_div

            signal = resample_poly(raw_signal, up, down)
            fs_out = float(target_rate_int)
            time = np.arange(len(signal)) / fs_out
            resampled = True
        else:
            signal = raw_signal.astype(float, copy=False)
            time = raw_time.astype(float, copy=False)
            fs_out = float(original_fs)
            resampled = False

        meta = {
            "h5_path": h5_path,
            "source_group": "resp_clean",
            "source_signal_key": "resp_clean/signal",
            "source_time_key": "resp_clean/time",
            "original_fs": float(original_fs),
            "target_rate": None if target_rate is None else float(target_rate),
            "final_fs": float(fs_out),
            "resampled": resampled,
            "n_samples": int(len(signal)),
            "duration_sec": float(time[-1] - time[0]) if len(time) > 1 else 0.0,
        }

        return signal, time, fs_out, meta

    except Exception as e:
        print(f"Error loading {h5_path}: {e}")
        return None, None, None, None

In [7]:
import os
import re
import glob
import numpy as np
import pandas as pd


# =========================================================
# 1) INPUTS
# =========================================================

cagemate_rank_h5s = r"C:\Users\thoma\UF Dropbox\Thomas Heeps\Padilla-Coreano Lab\2025\BLA_ChR_resp_pilot1\BLA_resp_ChR_cm_hc\h5_outputs"

rename_map = {
    "1_1_d_cm_1_2_i_20260106_134350.h5": "1_1_2_i_cm_1_2_d_20260106_134350_SUB.h5",
    "1_1_i_cm_1_3_s_20260107_163611.h5": "1_1_2_i_cm_1_3_s_20260107_163611_DOM.h5",
    "1_2_d_cm_1_3_s_20260108_114416.h5": "1_2_2_d_cm_1_3_s_20260108_114416_DOM.h5",
    "1_3_s_cm_1_1_i_20260106_122823.h5": "1_3_2_s_cm_1_1_i_20260106_122823_DOM.h5",
    "1_3_s_1_2_d_20260107_165340.h5": "1_3_2_s_cm_1_2_d_20260107_165340_SUB.h5",
    "2_1_d_cm_2.3_i_20260106_105109.h5": "2_1_2_d_cm_2.3_i_20260106_105109_SUB.h5",
    "2_1_d_cm_2_2_s_20260108_112426.h5": "2_1_2_d_cm_2_2_s_20260108_112426_DOM.h5",
    "2_2_s_cm_2_1_d_20260108_120453.h5": "2_2_2_s_cm_2_1_d_20260108_120453_SUB.h5",
    "2_2_s_cm_2_3_i_20260106_132549.h5": "2_2_2_s_cm_2_3_i_20260106_132549_DOM.h5",
    "3_1_i_cm_3_2_d_20260106_115157.h5": "3_1_2_i_cm_3_2_d_20260106_115157_SUB.h5",
    "3_1_i_cm_3_3_s_20260107_123230.h5": "3_1_2_i_cm_3_3_s_20260107_123230_DOM.h5",
    "3_2_d_cm_3_3_s_20260107_140924.h5": "3_2_2_d_cm_3_3_s_20260107_140924_DOM.h5",
    "3_3_s_cm_3_1_i_20260106_124546.h5": "3_3_2_s_cm_3_1_i_20260106_124546_DOM.h5",
    "3_3_s_cm_3_2_d_20260107_171330.h5": "3_3_2_s_cm_3_2_d_20260107_171330_SUB.h5",
    "4_1_s_cm_4_2_d_20260107_160439.h5": "4_1_2_s_cm_4_2_d_20260107_160439_SUB.h5",
    "4_2_d_cm_4_1_s_20260107_150836.h5": "4_2_2_d_cm_4_1_s_20260107_150836_DOM.h5",
    "4_3_d_cm_4_1_s_20260107_113120.h5": "4_3_2_d_cm_4_1_s_20260107_113120_DOM.h5",
    "5_1_i_cm_5_2_d_20260106_120946.h5": "5_1_2_i_cm_5_2_d_20260106_120946_SUB.h5",
    "5_1_i_cm_5_3_s_20260107_132309.h5": "5_1_2_i_cm_5_3_s_20260107_132309_DOM.h5",
    "6_2_d_cm_6_3_s_20260107_154939.h5": "6_2_2_d_cm_6_3_s_20260107_154939_DOM.h5",
    "7_1_d_cm_7_2_s_20260106_111330.h5": "7_1_2_d_cm_7_2_s_20260106_111330_DOM.h5",
    "7_2_s_cm_7_1_d_20260106_130431.h5": "7_2_2_s_cm_7_1_d_20260106_130431_SUB.h5",
    "8_1_s_cm_8_2_d_20260107_152812.h5": "8_1_2_s_cm_8_2_d_20260107_152812_SUB.h5",
    "8_2_s_cm_8_1_d_20260106_113241.h5": "8_2_2_s_cm_8_1_d_20260106_113241_SUB.h5",
}


# =========================================================
# 2) HELPERS
# =========================================================

def list_cagemate_rank_files(base_dir, rename_map, skip_rec_h5=True, require_in_map=True):
    """
    Build a dictionary of raw filename -> full path for the cagemate rank dataset.

    Why this exists:
    - In the valence pipeline you already had dictionaries of paths.
    - Here you only have a directory, so we need to discover files first.
    - We optionally skip '.rec.h5' files because those are usually duplicate or
      intermediate outputs, not the main recording we want to classify.

    Parameters
    ----------
    base_dir : str
        Folder containing the raw .h5 files.
    rename_map : dict
        Maps original raw filenames to standardized filenames with DOM/SUB suffixes.
    skip_rec_h5 : bool
        If True, skip files ending with '.rec.h5'.
    require_in_map : bool
        If True, only keep files that appear in rename_map. This is safer because
        the rank label comes from the renamed filename.
    """
    all_h5_paths = sorted(glob.glob(os.path.join(base_dir, "*.h5")))

    discovered = {}
    skipped = []

    for h5_path in all_h5_paths:
        raw_name = os.path.basename(h5_path)

        # Skip duplicate/intermediate rec exports
        if skip_rec_h5 and raw_name.endswith(".rec.h5"):
            skipped.append((raw_name, "skipped .rec.h5"))
            continue

        # If we require rename_map membership, skip unknown files
        if require_in_map and raw_name not in rename_map:
            skipped.append((raw_name, "not found in rename_map"))
            continue

        discovered[raw_name] = h5_path

    print(f"Found {len(discovered)} usable cagemate rank files")
    if skipped:
        print("\nSkipped files:")
        for fname, reason in skipped:
            print(f"  - {fname} ({reason})")

    return discovered


def parse_rank_metadata_from_names(raw_name, standardized_name):
    """
    Parse metadata from the original filename and the renamed standardized filename.

    Example standardized filename:
        1_1_2_i_cm_1_2_d_20260106_134350_SUB.h5

    We use the standardized filename because it contains the final label:
        ..._DOM.h5 or ..._SUB.h5

    Returns a dict with:
    - Recording: original file basename
    - StandardizedRecording: renamed file basename
    - PairID: cage/group pair, usually first number
    - Subject: recorded subject ID, e.g. '1_1'
    - Cagemate: other mouse ID, e.g. '1_2'
    - SubjectRankCode: original rank code of recorded mouse (d/i/s)
    - CagemateRankCode: original rank code of cagemate
    - RankLabel: DOM or SUB, the classification target
    - TrialKey: convenient stable key for grouping
    """
    std_no_ext = standardized_name.replace(".h5", "")

    # Pattern based on your renamed convention:
    # pair_subjectnum_2_subjectrank_cm_pair_cagematenum_cagematerank_date_time_label
    #
    # Example:
    # 1_1_2_i_cm_1_2_d_20260106_134350_SUB
    #
    # group(1) = pair id for recorded mouse
    # group(2) = recorded mouse number
    # group(3) = recorded mouse rank code
    # group(4) = pair id for cagemate
    # group(5) = cagemate mouse number
    # group(6) = cagemate rank code
    # group(7) = date
    # group(8) = time
    # group(9) = DOM or SUB
    pattern = (
        r"^(\d+)_([\d\.]+)_2_([dis])_cm_(\d+)_([\d\.]+)_([dis])_"
        r"(\d{8})_(\d{6})_(DOM|SUB)$"
    )

    m = re.match(pattern, std_no_ext, flags=re.IGNORECASE)
    if not m:
        # Fall back to something minimal so the row is still usable
        return {
            "Recording": raw_name,
            "StandardizedRecording": standardized_name,
            "PairID": np.nan,
            "Subject": np.nan,
            "Cagemate": np.nan,
            "SubjectRankCode": np.nan,
            "CagemateRankCode": np.nan,
            "RankLabel": np.nan,
            "TrialKey": os.path.splitext(raw_name)[0],
        }

    pair_id_subject = str(m.group(1))
    subject_num = str(m.group(2))
    subject_rank_code = str(m.group(3)).lower()

    pair_id_cm = str(m.group(4))
    cagemate_num = str(m.group(5))
    cagemate_rank_code = str(m.group(6)).lower()

    date_str = str(m.group(7))
    time_str = str(m.group(8))
    rank_label = str(m.group(9)).upper()

    subject_id = f"{pair_id_subject}_{subject_num}"
    cagemate_id = f"{pair_id_cm}_{cagemate_num}"

    # TrialKey is useful for grouping windows from the same session later
    trial_key = f"{subject_id}_cm_{cagemate_id}_{date_str}_{time_str}"

    return {
        "Recording": raw_name,
        "StandardizedRecording": standardized_name,
        "PairID": pair_id_subject,
        "Subject": subject_id,
        "Cagemate": cagemate_id,
        "SubjectRankCode": subject_rank_code,
        "CagemateRankCode": cagemate_rank_code,
        "RankLabel": rank_label,
        "TrialKey": trial_key,
    }


def code_to_rank_name(rank_code):
    """
    Converts shorthand rank code into a readable label.

    Based on your filenames:
    - d = dominant
    - s = subordinate
    - i = intermediate

    This is metadata only, not the classification target.
    The target should be RankLabel (DOM/SUB) from the rename_map standardization.
    """
    if pd.isna(rank_code):
        return np.nan

    rank_code = str(rank_code).lower()
    lookup = {
        "d": "dominant",
        "i": "intermediate",
        "s": "subordinate",
    }
    return lookup.get(rank_code, rank_code)


# =========================================================
# 3) MAIN PIPELINE
# =========================================================

def build_cagemate_rank_feature_matrix(
    base_dir,
    rename_map,
    data_type="rodentAirflow",
    target_srate=400,
    window_sec=3.0,
    n_windows_per_session=100,
    min_breaths_per_window=5,
    random_state=42,
    skip_rec_h5=True,
    require_in_map=True,
):
    """
    Build a window-level feature matrix for the cagemate rank dataset.

    This follows the same logic as your valence pipeline:
    1. Discover files from a directory
    2. Load full respiration signal
    3. Fit BreathMetrics ONCE on the full session
    4. Sample random non-overlapping windows from the session
    5. Extract window-level features using the session BreathMetrics object
    6. Add metadata for downstream classification

    Important design choice:
    - This does NOT refit BreathMetrics separately on each small window.
    - It assumes your existing `extract_features_from_windows(...)` function
      filters the full-session BM breath events down to each window.
    - That is typically more stable than fitting BM independently on tiny snippets.
    """
    all_rows = []

    raw_to_path = list_cagemate_rank_files(
        base_dir=base_dir,
        rename_map=rename_map,
        skip_rec_h5=skip_rec_h5,
        require_in_map=require_in_map,
    )

    if len(raw_to_path) == 0:
        print("No usable files found.")
        return pd.DataFrame()

    for raw_name, h5_path in raw_to_path.items():
        standardized_name = rename_map.get(raw_name, raw_name)
        meta_info = parse_rank_metadata_from_names(raw_name, standardized_name)

        print(f"\n[Rank] {raw_name}")
        print(f"  standardized -> {standardized_name}")

        # -------------------------------------------------
        # Load and preprocess respiration signal
        # -------------------------------------------------
        signal, time, fs, load_meta = load_clean_resp_signal_cagemate_rank(
            h5_path,
            target_rate=target_srate
        )

        if signal is None:
            print("  ❌ Load failed")
            continue

        if len(signal) == 0 or len(time) == 0:
            print("  ❌ Empty signal after preprocessing")
            continue

        # -------------------------------------------------
        # Fit BreathMetrics to the FULL session
        # -------------------------------------------------
        bm = fit_bm_session(signal, fs, data_type=data_type)
        if bm is None:
            print("  ❌ BreathMetrics fit failed")
            continue

        n_breaths_total = len(getattr(bm, "inhaleOnsets", []))
        print(f"  ✓ {n_breaths_total} breaths detected in full session")

        # -------------------------------------------------
        # Sample random non-overlapping windows
        # -------------------------------------------------
        windows = sample_random_nonoverlapping_windows(
            time=time,
            signal=signal,
            window_dur=window_sec,
            n_windows=n_windows_per_session,
            seed=random_state,
            allow_partial_if_short=False,
        )

        if len(windows) == 0:
            print("  ⚠️ No usable full windows")
            continue

        # -------------------------------------------------
        # Extract features per window using BM events
        # -------------------------------------------------
        window_rows = extract_features_from_windows(
            bm=bm,
            windows=windows,
            min_breaths_per_window=min_breaths_per_window,
        )

        if len(window_rows) == 0:
            print("  ⚠️ No usable windows after min-breath filtering")
            continue

        print(f"  ✓ {len(window_rows)} usable windows")

        # -------------------------------------------------
        # Add metadata to each window row
        # -------------------------------------------------
        session_duration_sec = float(time[-1] - time[0]) if len(time) > 1 else np.nan

        for row in window_rows:
            row.update({
                # IDs / filenames
                "Trial": meta_info["TrialKey"],
                "Recording": meta_info["Recording"],
                "StandardizedRecording": meta_info["StandardizedRecording"],

                # subject metadata
                "Subject": meta_info["Subject"],             # recorded mouse
                "Cagemate": meta_info["Cagemate"],           # partner mouse
                "PairID": meta_info["PairID"],

                # original rank codes from filename
                "SubjectRankCode": meta_info["SubjectRankCode"],
                "CagemateRankCode": meta_info["CagemateRankCode"],

                # readable rank metadata
                "SubjectRankName": code_to_rank_name(meta_info["SubjectRankCode"]),
                "CagemateRankName": code_to_rank_name(meta_info["CagemateRankCode"]),

                # classification target
                "Condition": meta_info["RankLabel"],         # keep same style as valence pipeline
                "RankLabel": meta_info["RankLabel"],         # explicit target column
                "Type": "CagemateRank",

                # session-level bookkeeping
                "session_duration_sec": session_duration_sec,
                "n_breaths_total": n_breaths_total,
                "WindowSec": window_sec,
                "TargetSRate": target_srate,
                "MinBreathsPerWindow": min_breaths_per_window,
            })

            all_rows.append(row)

    # -----------------------------------------------------
    # Assemble final DataFrame
    # -----------------------------------------------------
    if not all_rows:
        print("\n⚠️ No rows collected.")
        return pd.DataFrame()

    master_df = pd.DataFrame(all_rows).reset_index(drop=True)

    print(f"\n✅ Cagemate rank feature matrix: {len(master_df)} rows × {len(master_df.columns)} cols")
    print(f"   Recordings : {master_df['Recording'].nunique()}")
    print(f"   Subjects   : {master_df['Subject'].nunique()}")
    print(f"   Cagemates  : {master_df['Cagemate'].nunique()}")
    print(f"   Pairs      : {master_df['PairID'].nunique()}")
    print(f"   Rank dist  :\n{master_df['RankLabel'].value_counts(dropna=False)}")

    return master_df


# =========================================================
# 4) RUN IT
# =========================================================

rank_feature_df = build_cagemate_rank_feature_matrix(
    base_dir=cagemate_rank_h5s,
    rename_map=rename_map,
    data_type="rodentAirflow",
    target_srate=400,
    window_sec=3.0,
    n_windows_per_session=100,
    min_breaths_per_window=5,
    random_state=42,
    skip_rec_h5=True,
    require_in_map=True,
)

rank_feature_df.head()

Found 24 usable cagemate rank files

Skipped files:
  - 2_2_s_cm_2_1_d_20260108_120453.rec.h5 (skipped .rec.h5)

[Rank] 1_1_d_cm_1_2_i_20260106_134350.h5
  standardized -> 1_1_2_i_cm_1_2_d_20260106_134350_SUB.h5
  ✓ 4136 breaths detected in full session
  ✓ 98 usable windows

[Rank] 1_1_i_cm_1_3_s_20260107_163611.h5
  standardized -> 1_1_2_i_cm_1_3_s_20260107_163611_DOM.h5
  ✓ 4857 breaths detected in full session
  ✓ 100 usable windows

[Rank] 1_2_d_cm_1_3_s_20260108_114416.h5
  standardized -> 1_2_2_d_cm_1_3_s_20260108_114416_DOM.h5
  ✓ 3997 breaths detected in full session
  ✓ 99 usable windows

[Rank] 1_3_s_1_2_d_20260107_165340.h5
  standardized -> 1_3_2_s_cm_1_2_d_20260107_165340_SUB.h5
  ✓ 3402 breaths detected in full session
  ✓ 95 usable windows

[Rank] 1_3_s_cm_1_1_i_20260106_122823.h5
  standardized -> 1_3_2_s_cm_1_1_i_20260106_122823_DOM.h5
  ✓ 4398 breaths detected in full session
  ✓ 100 usable windows

[Rank] 2_1_d_cm_2.3_i_20260106_105109.h5
  standardized -> 2_1_2_d_c

,WindowID,Start,Stop,Duration,n_breaths,breathing_rate_hz,mean_ibi_sec,cv_ibi,mean_inhale_dur_sec,cv_inhale_dur,mean_exhale_dur_sec,cv_exhale_dur,ie_ratio,mean_peak_insp_flow,mean_peak_exp_flow,cv_peak_insp_flow,mean_inhale_vol,mean_exhale_vol,mean_tidal_vol_proxy,cv_inhale_vol,ventilation_proxy,pct_breaths_with_inhale_pause,mean_inhale_pause_dur_sec,cv_inhale_pause_dur,inhale_pause_duty_cycle,pct_breaths_with_exhale_pause,mean_exhale_pause_dur_sec,cv_exhale_pause_dur,exhale_pause_duty_cycle,inhale_duty_cycle,exhale_duty_cycle,Trial,Recording,StandardizedRecording,Subject,Cagemate,PairID,SubjectRankCode,CagemateRankCode,SubjectRankName,CagemateRankName,Condition,RankLabel,Type,session_duration_sec,n_breaths_total,WindowSec,TargetSRate,MinBreathsPerWindow
0,0,0.0,3.0,3.0,21,6.950478,0.143875,0.908652,0.089286,1.455465,0.049643,0.398139,1.798561,1006.565044,-541.945195,0.205289,55503.536613,19579.581888,75083.118501,1.288202,521863.551704,0.0,0.0,NaN,0.0,0.0,0.0,NaN,0.0,0.620578,0.345042,1_1_cm_1_2_20260106_134350,1_1_d_cm_1_2_i_20260106_134350.h5,1_1_2_i_cm_1_2_d_20260106_134350_SUB.h5,1_1,1_2,1,i,d,intermediate,dominant,SUB,SUB,CagemateRank,610.2675,4136,3.0,400,5
1,1,12.0,15.0,3.0,32,10.634648,0.094032,0.157024,0.041953,0.127053,0.048359,0.351955,0.867528,750.305747,-594.269205,0.246527,22222.771366,20116.862471,42339.633836,0.322271,450267.117986,0.0,0.0,NaN,0.0,0.0,0.0,NaN,0.0,0.446157,0.514285,1_1_cm_1_2_20260106_134350,1_1_d_cm_1_2_i_20260106_134350.h5,1_1_2_i_cm_1_2_d_20260106_134350_SUB.h5,1_1,1_2,1,i,d,intermediate,dominant,SUB,SUB,CagemateRank,610.2675,4136,3.0,400,5
2,2,18.0,21.0,3.0,28,9.129332,0.109537,0.267107,0.048750,0.366099,0.054911,0.440992,0.887805,754.440207,-552.051235,0.342122,25631.799068,21504.547945,47136.347013,0.456576,430323.370875,0.0,0.0,NaN,0.0,0.0,0.0,NaN,0.0,0.445055,0.501298,1_1_cm_1_2_20260106_134350,1_1_d_cm_1_2_i_20260106_134350.h5,1_1_2_i_cm_1_2_d_20260106_134350_SUB.h5,1_1,1_2,1,i,d,intermediate,dominant,SUB,SUB,CagemateRank,610.2675,4136,3.0,400,5
3,3,21.0,24.0,3.0,26,8.673027,0.115300,0.318509,0.046827,0.428017,0.062500,0.544624,0.749231,636.928249,-494.556062,0.268867,20504.181949,23075.530753,43579.712701,0.386848,377968.019959,0.0,0.0,NaN,0.0,0.0,0.0,NaN,0.0,0.406131,0.542064,1_1_cm_1_2_20260106_134350,1_1_d_cm_1_2_i_20260106_134350.h5,1_1_2_i_cm_1_2_d_20260106_134350_SUB.h5,1_1,1_2,1,i,d,intermediate,dominant,SUB,SUB,CagemateRank,610.2675,4136,3.0,400,5
4,4,27.0,30.0,3.0,29,9.638554,0.103750,0.386750,0.051724,0.786462,0.046466,0.214989,1.113173,819.693061,-560.851178,0.275486,27888.309553,17602.464152,45490.773705,0.551187,438465.288723,0.0,0.0,NaN,0.0,0.0,0.0,NaN,0.0,0.498546,0.447860,1_1_cm_1_2_20260106_134350,1_1_d_cm_1_2_i_20260106_134350.h5,1_1_2_i_cm_1_2_d_20260106_134350_SUB.h5,1_1,1_2,1,i,d,intermediate,dominant,SUB,SUB,CagemateRank,610.2675,4136,3.0,400,5


In [8]:
rank_feature_df['n_breaths'].min()

np.int64(5)

In [9]:
rank_feature_df['n_breaths'].max()

np.int64(33)

In [10]:
# pickle
rank_feature_df.to_pickle(r"C:\Users\thoma\Code\ResearchCode\respiratory_pilot\notebooks\thomas_notebooks\resp_classification\data\raw-breathmetrics-features-session-3sec-rank")